In [1]:
import os
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
os.makedirs("data", exist_ok=True)

hospitals = pd.DataFrame({
    "hospital_id": range(1, 7),
    "hospital_name": ["Riverside General", "Summit Heart Institute", "Harbor Medical Center",
                      "Cedar Teaching Hospital", "Northgate Community", "Lakeview Specialty"],
    "region": ["Central", "North", "Coastal", "Central", "North", "Coastal"],
    "type": ["General", "Specialty", "General", "Teaching", "Community", "Specialty"],
    "beds": [450, 180, 320, 600, 220, 150],
})

departments = pd.DataFrame({
    "department_id": range(1, 9),
    "department": ["Emergency", "Cardiology", "Internal Medicine", "Surgery",
                   "Orthopedics", "Pediatrics", "Obstetrics", "Oncology"],
})

diagnoses = pd.DataFrame({
    "diagnosis_id": range(1, 13),
    "icd10_code": ["I21", "I50", "E11", "J21", "J44", "J18", "N18", "K35", "S72", "C34", "O80", "A41"],
    "diagnosis": ["Acute myocardial infarction", "Heart failure", "Type 2 diabetes", "Bronchiolitis",
                  "COPD", "Pneumonia", "Chronic kidney disease", "Acute appendicitis",
                  "Femur fracture", "Lung cancer", "Normal delivery", "Sepsis"],
    "department_id": [2, 2, 3, 6, 3, 3, 3, 4, 5, 8, 7, 1],
    "base_los_days": [5, 6, 4, 3, 5, 5, 6, 3, 9, 7, 2, 10],
    "base_cost": [18000, 14000, 6000, 4500, 7500, 8000, 9000, 9500, 22000, 25000, 3500, 30000],
})

n = 40000
age = np.clip(rng.gamma(4, 12, n), 0, 95).astype(int)
patients = pd.DataFrame({
    "patient_id": range(1, n + 1),
    "gender": rng.choice(["F", "M"], n),
    "birth_year": 2025 - age,
    "smoker": ((rng.random(n) < 0.18) & (age >= 16)).astype(int),
    "has_diabetes": (rng.random(n) < 0.02 + age * 0.003).astype(int),
    "has_hypertension": (rng.random(n) < 0.05 + age * 0.006).astype(int),
})

for name, df in [("dim_hospital", hospitals), ("dim_department", departments),
                 ("dim_diagnosis", diagnoses), ("dim_patient", patients)]:
    df.to_csv(f"data/{name}.csv", index=False)
    print(name, len(df))

dim_hospital 6
dim_department 8
dim_diagnosis 12
dim_patient 40000


In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
hospitals = pd.read_csv("data/dim_hospital.csv")
diagnoses = pd.read_csv("data/dim_diagnosis.csv")
patients = pd.read_csv("data/dim_patient.csv")
patients["age"] = 2025 - patients["birth_year"]

N = 200_000

# --- Dates: 6 years, winter peak in mid-January
days = pd.date_range("2020-01-01", "2025-12-31")
season = np.cos(2 * np.pi * (days.dayofyear.values - 15) / 365)
p_day = 1 + 0.25 * season
p_day /= p_day.sum()
day_idx = rng.choice(len(days), N, p=p_day)
admit_date = days[day_idx]
season_a = season[day_idx]
year = admit_date.year.values
month = admit_date.month.values
winter = np.isin(month, [12, 1, 2])

# --- Patients: older and chronically ill patients are admitted more often
w = 1 + patients["age"] / 30 + patients["has_diabetes"] + patients["has_hypertension"] + (patients["age"] <= 5) * 10
pt = patients.iloc[rng.choice(len(patients), N, p=(w / w.sum()).values)].reset_index(drop=True)
age = pt["age"].values
female = (pt["gender"] == "F").values
diab = pt["has_diabetes"].values
htn = pt["has_hypertension"].values

# --- Hospitals: by bed count. FINDING 2: hospital 3 gets 60% more admissions in winter
hw = np.tile(hospitals["beds"].values.astype(float), (N, 1))
hw[winter, 2] *= 1.6
hw /= hw.sum(axis=1, keepdims=True)
hosp = np.minimum((hw.cumsum(axis=1) < rng.random(N)[:, None]).sum(axis=1), 5)

# --- Diagnoses: depend on age, sex, diabetes, season
W = np.ones((N, 12))
older = np.clip(age / 50, 0, 2.5) ** 2
W[:, [0, 1, 4, 6, 9]] *= older[:, None]
W[:, 2] *= 1 + 4 * diab
W[:, 3] = np.where(age <= 5, 8, 0)
W[:, 8] *= np.where(age > 60, 3, 0.3)
W[:, 10] = np.where(female & (age >= 18) & (age <= 40), 4, 0)
W[:, [3, 4, 5]] *= (1 + 0.8 * np.clip(season_a, 0, 1))[:, None]
W /= W.sum(axis=1, keepdims=True)
dx = np.minimum((W.cumsum(axis=1) < rng.random(N)[:, None]).sum(axis=1), 11)

dx_los = diagnoses["base_los_days"].values[dx]
dx_cost = diagnoses["base_cost"].values[dx]
er_p = np.array([1.0, .7, .4, .9, .7, .8, .3, 1.0, .9, .15, .5, 1.0])
mort = np.array([.06, .05, .005, .002, .03, .04, .03, .002, .03, .12, .0003, .2])
readm = np.array([.3, .8, .3, 0, .6, .2, .4, -1, -.3, .2, -1, .2])

# --- Labs
hba1c = np.where(diab == 1, np.clip(rng.normal(7.8, 1.4, N), 5, 13), np.nan)
poor_control = hba1c >= 8
creat = np.clip(rng.normal(0.9, 0.2, N) + 0.004 * np.clip(age - 40, 0, None)
                + np.where(dx == 6, rng.normal(1.8, 0.6, N), 0), 0.4, 9)
crp = np.clip(rng.gamma(2, 4, N) * np.where(np.isin(dx, [5, 11]), 6, 1), 0.5, 400)

# --- Length of stay, type, cost
los = (dx_los * rng.lognormal(0, 0.3, N) * (1 + np.clip(age - 50, 0, None) / 100)
       * (1 + 0.15 * diab + 0.1 * htn) * np.where(poor_control, 1.15, 1)
       * np.where((hosp == 2) & winter, 1.2, 1))
los = np.maximum(1, np.round(los)).astype(int)
is_er = rng.random(N) < er_p[dx]

# FINDING 3: poor diabetes control (HbA1c >= 8) adds 25% to cost
cost = (dx_cost * (0.5 + 0.5 * los / dx_los) * np.where(is_er, 1.2, 1)
        * np.where(poor_control, 1.25, 1) * (1 + 0.04 * (year - 2020))
        * rng.lognormal(0, 0.15, N))

# --- Outcomes
died = rng.random(N) < np.clip(mort[dx] * (1 + np.clip(age - 60, 0, None) / 40), 0, 0.9)

# FINDING 1: hospital 5 readmissions rise each year
logit = (-2.6 + 0.02 * (age - 50) + 0.4 * diab + 0.3 * htn + 0.04 * los + readm[dx]
         + np.where(hosp == 4, 0.15 * (year - 2020), 0))
readmitted = (rng.random(N) < 1 / (1 + np.exp(-logit))) & ~died

# --- Assemble and save
df = pd.DataFrame({
    "patient_id": pt["patient_id"].values,
    "hospital_id": hospitals["hospital_id"].values[hosp],
    "department_id": diagnoses["department_id"].values[dx],
    "diagnosis_id": diagnoses["diagnosis_id"].values[dx],
    "admit_date": admit_date,
    "discharge_date": admit_date + pd.to_timedelta(los, unit="D"),
    "length_of_stay": los,
    "admission_type": np.where(is_er, "Emergency", "Elective"),
    "total_cost": cost.round(2),
    "outcome": np.where(died, "Deceased", "Discharged"),
    "readmitted_30d": readmitted.astype(int),
    "hba1c": hba1c, "creatinine": creat, "crp": crp,
})
df = df.sort_values("admit_date").reset_index(drop=True)
df.insert(0, "admission_id", df.index + 1)

lab_cols = ["hba1c", "creatinine", "crp"]
admissions = df.drop(columns=lab_cols)
labs = df.melt(id_vars="admission_id", value_vars=lab_cols,
               var_name="test_name", value_name="value").dropna()
labs["value"] = labs["value"].round(2)
labs["unit"] = labs["test_name"].map({"hba1c": "%", "creatinine": "mg/dL", "crp": "mg/L"})
labs.insert(0, "lab_id", np.arange(1, len(labs) + 1))

admissions.to_csv("data/fact_admissions.csv", index=False)
labs.to_csv("data/fact_lab_results.csv", index=False)

print(len(admissions), len(labs))
print(admissions.groupby(["hospital_id", admissions["admit_date"].dt.year])["readmitted_30d"]
      .mean().unstack().round(3))

200000 444148
admit_date    2020   2021   2022   2023   2024   2025
hospital_id                                          
1            0.142  0.153  0.148  0.150  0.153  0.153
2            0.136  0.145  0.148  0.166  0.144  0.134
3            0.149  0.154  0.159  0.151  0.157  0.149
4            0.150  0.149  0.150  0.155  0.150  0.150
5            0.160  0.173  0.190  0.203  0.229  0.262
6            0.152  0.144  0.154  0.158  0.142  0.146


In [4]:
import pandas as pd

a = pd.read_csv("data/fact_admissions.csv", parse_dates=["admit_date"])
l = pd.read_csv("data/fact_lab_results.csv")

# Finding 2: share of each hospital's admissions that fall in winter
a["winter"] = a["admit_date"].dt.month.isin([12, 1, 2])
print(a.groupby("hospital_id")["winter"].mean().round(3))

# Finding 3: average cost, diabetic patients by HbA1c control
h = l[l["test_name"] == "hba1c"].merge(a[["admission_id", "total_cost"]], on="admission_id")
print(h.groupby(h["value"] >= 8)["total_cost"].mean().round(0))

hospital_id
1    0.281
2    0.283
3    0.384
4    0.282
5    0.287
6    0.287
Name: winter, dtype: float64
value
False    19097.0
True     26350.0
Name: total_cost, dtype: float64
